In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from mealpy.swarm_based import PSO
from mealpy.utils.problem import FloatVar
import warnings
warnings.filterwarnings('ignore')

# ----------------------------
# Load your Sentiment dataset
# ----------------------------
def load_data():
    amazon = pd.read_csv("../sentiment labelled sentences/amazon_cells_labelled.txt", sep="\t", header=None, names=["text", "label"])
    imdb   = pd.read_csv("../sentiment labelled sentences/imdb_labelled.txt", sep="\t", header=None, names=["text", "label"])
    yelp   = pd.read_csv("../sentiment labelled sentences/yelp_labelled.txt", sep="\t", header=None, names=["text", "label"])

    df = pd.concat([amazon, imdb, yelp], axis=0)

    vectorizer = TfidfVectorizer(stop_words="english", max_features=1000)
    X = vectorizer.fit_transform(df["text"]).toarray()
    y = df["label"].values

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    return X_train, X_test, y_train, y_test

# ----------------------------
# Objective function for PSO
# ----------------------------
def objective_function(solution):
    global X_train, y_train

    n_estimators = int(solution[0])
    max_depth = int(solution[1]) if solution[1] > 0 else None
    min_samples_split = int(solution[2])
    min_samples_leaf = int(solution[3])
    max_features = solution[4]

    try:
        rf = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            max_features=max_features,
            random_state=42,
            n_jobs=-1
        )
        scores = cross_val_score(rf, X_train, y_train, cv=3, scoring='accuracy')
        fitness = -np.mean(scores)  # Negative because optimizer minimizes
    except Exception:
        fitness = 1.0  # Penalty if invalid

    return fitness

# ----------------------------
# Run PSO optimization
# ----------------------------
def optimize_random_forest():
    global X_train, X_test, y_train, y_test
    X_train, X_test, y_train, y_test = load_data()

    print("Starting Random Forest optimization with Mealpy PSO...")
    print(f"Training set size: {X_train.shape}")
    print(f"Test set size: {X_test.shape}")
    print("-" * 50)

    # Problem bounds for hyperparameters
    problem = {
        "bounds": [
            FloatVar(lb=10, ub=200, name="n_estimators"),
            FloatVar(lb=1, ub=20, name="max_depth"),
            FloatVar(lb=2, ub=20, name="min_samples_split"),
            FloatVar(lb=1, ub=10, name="min_samples_leaf"),
            FloatVar(lb=0.1, ub=1.0, name="max_features")
        ],
        "minmax": "min",
        "obj_func": objective_function
    }

    optimizer = PSO.OriginalPSO(epoch=5, pop_size=10)
    best_agent = optimizer.solve(problem)  # Returns Agent object
    best_position = best_agent.solution          # Correct attribute for params
    best_fitness = best_agent.target.fitness     # Extract numeric float

    print("\nOptimization Results:")
    print("-" * 50)
    print(f"Best fitness (negative accuracy): {best_fitness:.6f}")
    print(f"Best accuracy: {-best_fitness:.6f}")

    best_params = {
        'n_estimators': int(best_position[0]),
        'max_depth': int(best_position[1]) if best_position[1] > 0 else None,
        'min_samples_split': int(best_position[2]),
        'min_samples_leaf': int(best_position[3]),
        'max_features': best_position[4],
        'random_state': 42
    }

    print("\nBest Hyperparameters:")
    for param, value in best_params.items():
        print(f"  {param}: {value}")

    return best_params

# ----------------------------
# Evaluate optimized model
# ----------------------------
def evaluate_model(best_params):
    global X_train, X_test, y_train, y_test

    print("\n" + "="*50)
    print("FINAL MODEL EVALUATION")
    print("="*50)

    # Ensure required hyperparameters exist; fall back to safe defaults if missing
    # Normalize max_depth
    max_depth_val = best_params.get('max_depth', None)
    if max_depth_val in (None, 'None'):
        max_depth = None
    else:
        max_depth = int(max_depth_val)

    # Other integer hyperparameters with safe defaults
    n_estimators = int(best_params.get('n_estimators', 100))
    min_samples_split = int(best_params.get('min_samples_split', 2))
    min_samples_leaf = int(best_params.get('min_samples_leaf', 1))

    # Normalize max_features (allow float fraction or string like 'sqrt')
    max_features_val = best_params.get('max_features', 'sqrt')
    try:
        if isinstance(max_features_val, (float, np.floating, int, np.integer)):
            max_features = float(max_features_val)
        else:
            max_features = max_features_val
    except Exception:
        max_features = 'sqrt'

    # Ensure random seed is provided
    random_state = int(best_params.get('random_state', 42))

    # Build the final RandomForestClassifier with explicit hyperparameters
    best_rf = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        random_state=random_state,
        n_jobs=-1
    )
    best_rf.fit(X_train, y_train)

    y_pred = best_rf.predict(X_test)
    test_accuracy = accuracy_score(y_test, y_pred)

    print(f"Test Accuracy: {test_accuracy:.6f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))

    print("\nComparison with default Random Forest:")
    # Provide explicit seed and include min_samples_leaf and max_features for reproducibility
    # Also set n_jobs for faster execution
    default_rf = RandomForestClassifier(
        n_estimators=100,
        min_samples_leaf=1,
        max_features='sqrt',
        random_state=42,
        n_jobs=-1
    )
    default_rf.fit(X_train, y_train)
    default_pred = default_rf.predict(X_test)
    default_accuracy = accuracy_score(y_test, default_pred)

    print(f"Default RF Accuracy: {default_accuracy:.6f}")
    print(f"Optimized RF Accuracy: {test_accuracy:.6f}")
    print(f"Improvement: {test_accuracy - default_accuracy:.6f}")

if __name__ == "__main__":
    best_params = optimize_random_forest()
    evaluate_model(best_params)
    print("\n" + "="*50)
    print("OPTIMIZATION COMPLETE!")
    print("="*50)

2025/10/10 06:41:16 PM, INFO, mealpy.swarm_based.PSO.OriginalPSO: OriginalPSO(epoch=5, pop_size=10, c1=2.05, c2=2.05, w=0.4)


Starting Random Forest optimization with Mealpy PSO...
Training set size: (2198, 1000)
Test set size: (550, 1000)
--------------------------------------------------


2025/10/10 06:41:59 PM, INFO, mealpy.swarm_based.PSO.OriginalPSO: >>>Problem: P, Epoch: 1, Current best: -0.70701784467356, Global best: -0.70701784467356, Runtime: 25.26122 seconds
2025/10/10 06:42:23 PM, INFO, mealpy.swarm_based.PSO.OriginalPSO: >>>Problem: P, Epoch: 2, Current best: -0.70701784467356, Global best: -0.70701784467356, Runtime: 23.65298 seconds
2025/10/10 06:42:43 PM, INFO, mealpy.swarm_based.PSO.OriginalPSO: >>>Problem: P, Epoch: 3, Current best: -0.7102017310401897, Global best: -0.7102017310401897, Runtime: 20.02636 seconds
2025/10/10 06:43:04 PM, INFO, mealpy.swarm_based.PSO.OriginalPSO: >>>Problem: P, Epoch: 4, Current best: -0.7102017310401897, Global best: -0.7102017310401897, Runtime: 21.57947 seconds
2025/10/10 06:43:21 PM, INFO, mealpy.swarm_based.PSO.OriginalPSO: >>>Problem: P, Epoch: 5, Current best: -0.7102017310401897, Global best: -0.7102017310401897, Runtime: 17.09393 seconds



Optimization Results:
--------------------------------------------------
Best fitness (negative accuracy): -0.710202
Best accuracy: 0.710202

Best Hyperparameters:
  n_estimators: 94
  max_depth: 19
  min_samples_split: 13
  min_samples_leaf: 3
  max_features: 0.1372715213549638
  random_state: 42

FINAL MODEL EVALUATION
Test Accuracy: 0.734545

Classification Report:
              precision    recall  f1-score   support

    Negative       0.66      0.95      0.78       273
    Positive       0.91      0.53      0.67       277

    accuracy                           0.73       550
   macro avg       0.79      0.74      0.72       550
weighted avg       0.79      0.73      0.72       550


Comparison with default Random Forest:
Default RF Accuracy: 0.814545
Optimized RF Accuracy: 0.734545
Improvement: -0.080000

OPTIMIZATION COMPLETE!
